# WHaLE Example Usage and functionality

This notebook is purely for demonstration purposes on how to configure and use WHaLE. The first note is that a user's `library_path` should have three subfolders: "ORBIT", "WOMBAT", and "FLORIS", each setup in a way that aligns with the respective software's library specifications.

## Imports and environment set up

In [17]:
from time import perf_counter
from pathlib import Path

import pandas as pd

from whale import Project

%load_ext autoreload
%autoreload 2

pd.options.display.float_format = '{:,.4f}'.format 

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Create a `Project` from the individual components

**NOTE**: Make sure in the WOMBAT config files that the library path is the same that gets printed in this first code block when you run the notebook

In [18]:
# Define where the project data lives
library_path = Path("library").resolve()

# Define the file name for each simulation's configuration file
orbit_config = "Ocean_Wind_1_base_install.yaml"
wombat_config = "Ocean_Wind_1_base_operations.yaml"
floris_config = "Ocean_Wind_1_base_floris_jensen.yaml"
weather_file = "ocean_wind_1_39.0_-74.0_1959_2023.csv"  # ERA5 reanalysis profile, 1959-now

In [19]:
# Define the model
project = Project(
    library_path=library_path,
    weather_profile=weather_file,
    orbit_config=orbit_config,
    wombat_config=wombat_config,  # wombat_config if running full 20 year simulation
    floris_config=floris_config,
    
    # These allow us to connect the models appropriately
    orbit_weather_cols=["windspeed_10m", "windspeed_100m", "wave_height"],
    floris_windspeed="windspeed_100m",
    floris_wind_direction="wind_direction_100m",
    floris_x_col="floris_x",  # Column should exist, or can be created in the WOMBAT layout
    floris_y_col="floris_y",  # Column should exist, or can be created in the WOMBAT layout
)

In [21]:
project.wombat_config_dict["layout"]

'ocean_wind_1_layout_base.csv'

In [22]:
project.wombat.windfarm.layout_df.head()

,id,substation_id,name,type,latitude,longitude,string_actual,order_actual,string,order,distance,subassembly,upstream_cable,easting,northing,floris_x,floris_y
0,L107_A02,Substation Z01,L107_A02,turbine,39.2083,-74.1932,0.0000,2.0000,0.0000,0.0000,NaN,12MW_100pct_reduction.yaml,array.yaml,569658,4340200,15390,20167
1,L107_E05,Substation Z02,L107_E05,turbine,39.1321,-74.2298,0.0000,0.0000,0.0000,0.0000,NaN,12MW_100pct_reduction.yaml,array.yaml,566568,4331721,12300,11688
2,L107_H04,Substation Z11,L107_H04,turbine,39.1102,-74.2901,0.0000,0.0000,0.0000,0.0000,NaN,12MW_100pct_reduction.yaml,array.yaml,561380,4329242,7112,9209
3,L107_A03,Substation Z01,L107_A03,turbine,39.1987,-74.1812,0.0000,1.0000,0.0000,1.0000,NaN,12MW_100pct_reduction.yaml,array.yaml,570699,4339141,16431,19108
4,L107_E06,Substation Z02,L107_E06,turbine,39.1219,-74.2188,0.0000,1.0000,0.0000,1.0000,NaN,12MW_100pct_reduction.yaml,array.yaml,567530,4330594,13262,10561


## Additional Helper & Farm Visualization Methods

In [4]:
# # Method to create the distance-based x, y coordinates that will be used in FLORIS,
# # if they don't already exist, and reinitializes the FLORIS model
# project.generate_floris_positions_from_layout(
#     x_col="easting",
#     y_col="northing",
#     update_config=True,  # update the model and configuration
#     config_fname=floris_config,  # save over the original
# )

In [5]:
# fig, ax = project.plot_farm(return_fig=True)
# fig.savefig(library_path / "results" / "ocean_wind_layout.svg")

### Save the compiled configuration and reload it

In [6]:
# project.save_config("ocean_wind_example.yaml")  # saves to library_path / project / config
# project = Project.from_file(library_path, "ocean_wind_example.yaml")

## Run the analyses and calculate results

This separately calls the `run` methods for each of the ORBIT `ProjectManager` and WOMBAT `Simualation`, and the FLORIS wind rose or time series methods (dependent on the granularity of results desired), in that order. Alternatively, these could just be called on their own, if customized tinkering is desired.

In [7]:
# Run a wind rose based analysis for FLORIS for the analysis period

start = perf_counter()
project.run(
    which_floris="wind_rose",
    full_wind_rose=False,  # Create a wind rose from only the operational period
    floris_reinitialize_kwargs=dict(cut_in_wind_speed=3.0, cut_out_wind_speed=25.0)
)
end = perf_counter()
print(f"Run time: {(end - start) / 60:.2f} minutes")

Correcting negative Overhang:-2.5
Run time: 2.76 minutes


In [8]:
project.energy_production(frequency="project")

,wind_farm
L107_A02,"437,427.7301"
L107_E05,"648,359.4591"
L107_H04,"718,891.8037"
L107_A03,"10,987.8049"
L107_E06,"26,779.4722"
...,...
L107_I02,"940,716.6108"
L107_I01,"546,967.8208"
L107_J01,"6,635.0574"
L107_K01,"7,693.7546"


In [9]:
print(f"Unadjusted AEP: {project.aep_mwh / 1000:,.2f} GWh")

Unadjusted AEP: 4,888.29 GWh


In [10]:
# NOTE: THIS IS VERY LOW BECAUSE OF THE SETTINGS CURRENTLY IN PLACE!!

print(f"Realized AEP: {project.energy_production().values[0][0] / 1000:,.2f} GWh")

Realized AEP: 437.43 GWh


In [11]:
project.capex(per_mw=True, breakdown=True)

,CapEx,CapEx per MW
Array System,"67,608,092.0851","61,239.2138"
Export System,"431,889,698.0017","391,204.4366"
Offshore Substation,"171,240,450.0000","155,109.1033"
Scour Protection,"18,241,760.0000","16,523.3333"
Substructure,"514,120,861.3011","465,689.1860"
Array System Installation,"26,243,923.4050","23,771.6698"
Export System Installation,"213,964,802.8100","193,808.6982"
Offshore Substation Installation,"10,141,235.2939","9,185.9015"
Scour Protection Installation,"60,942,009.1324","55,201.0952"
Substructure Installation,"77,353,051.1013","70,066.1695"


In [12]:
project.npv(frequency="annual")

The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.FutureWarning: /opt/miniconda3/envs/whale/lib/python3.10/site-packages/wombat/core/post_processor.py:1738
The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.FutureWarning: /opt/miniconda3/envs/whale/lib/python3.10/site-packages/wombat/core/post_processor.py:788
The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.FutureWarning: /opt/miniconda3/envs/whale/lib/python3.10/site-packages/wombat/core/post_processor.py:1267
The default value of nu

,NPV
year,
2000,"-7,787,699.5345"
2001,"-7,648,818.7317"
2002,"-24,227,560.2597"
2003,"-29,934,104.7955"
2004,"-19,703,430.6839"
2005,"-11,749,983.5354"
2006,"-19,148,812.3819"
2007,"-25,481,610.5148"
2008,"-20,706,486.0432"


In [13]:
project.wombat.metrics.opex(frequency="project")

The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.

,OpEx
0,"363,231,958.6933"


In [14]:
project.wombat.env.cleanup_log_files()

In [15]:
# We can also compare the results for a time series based FLORIS analysis, but that takes ~90 minutes
# to run with the current example, and so is excluded for the sake of convenience

# project.run(
#     skip=["orbit", "wombat"],
#     which_floris="time_series",
#     cut_in_wind_speed=3.0,
#     cut_out_wind_speed=25.0,
#     nodes=8,
# )